In [0]:
#  Drift Detection
crm_df = spark.table("workspace.default.crm_silver")

billing_df = spark.table("workspace.default.billing_silver")

analytics_df = spark.table("workspace.default.analytics_silver")

In [0]:
crm_count = crm_df.count()

billing_count = billing_df.count()

analytics_count = analytics_df.count()

print("CRM Records :", crm_count)
print("Billing Records :", billing_count)
print("Analytics Records :", analytics_count)

CRM Records : 10516
Billing Records : 11692
Analytics Records : 912


In [0]:
volume_difference = abs(crm_count - billing_count)

print("Volume Difference:", volume_difference)

Volume Difference: 1176


In [0]:
volume_drift = (volume_difference / crm_count) * 100

print("Volume Drift Percentage:", round(volume_drift, 2), "%")

Volume Drift Percentage: 11.18 %


In [0]:
if volume_drift > 10:
    print("🚨 ALERT: Volume Drift Detected")
else:
    print("✅ No Significant Volume Drift")

🚨 ALERT: Volume Drift Detected


In [0]:
print("CRM Schema")
crm_df.printSchema()

CRM Schema
root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- city: string (nullable = true)



In [0]:
print("Billing Schema")
billing_df.printSchema()

Billing Schema
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- status: string (nullable = true)



In [0]:
print("Analytics Schema")
analytics_df.printSchema()

Analytics Schema
root
 |-- date: date (nullable = true)
 |-- total_customers: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- avg_transaction: double (nullable = true)



In [0]:
print("CRM Columns")
print(crm_df.columns)

print("\nBilling Columns")
print(billing_df.columns)

print("\nAnalytics Columns")
print(analytics_df.columns)

CRM Columns
['customer_id', 'name', 'email', 'signup_date', 'city']

Billing Columns
['transaction_id', 'customer_id', 'amount', 'transaction_date', 'status']

Analytics Columns
['date', 'total_customers', 'total_revenue', 'avg_transaction']


In [0]:
print("CRM Data Types")
print(crm_df.dtypes)

print("\nBilling Data Types")
print(billing_df.dtypes)

print("\nAnalytics Data Types")
print(analytics_df.dtypes)

CRM Data Types
[('customer_id', 'string'), ('name', 'string'), ('email', 'string'), ('signup_date', 'date'), ('city', 'string')]

Billing Data Types
[('transaction_id', 'string'), ('customer_id', 'string'), ('amount', 'double'), ('transaction_date', 'date'), ('status', 'string')]

Analytics Data Types
[('date', 'date'), ('total_customers', 'bigint'), ('total_revenue', 'double'), ('avg_transaction', 'double')]


In [0]:
crm_schema = set(crm_df.dtypes)
billing_schema = set(billing_df.dtypes)

schema_difference = crm_schema.symmetric_difference(billing_schema)

if len(schema_difference) == 0:
    print("✅ No Schema Drift Detected")
else:
    print("🚨 Schema Drift Detected")
    print(schema_difference)

🚨 Schema Drift Detected
{('transaction_id', 'string'), ('amount', 'double'), ('email', 'string'), ('name', 'string'), ('status', 'string'), ('signup_date', 'date'), ('transaction_date', 'date'), ('city', 'string')}


In [0]:
from pyspark.sql.functions import avg, min, max

billing_df.select(
    avg("amount").alias("Average Amount"),
    min("amount").alias("Minimum Amount"),
    max("amount").alias("Maximum Amount")
).show()

+-----------------+--------------+--------------+
|   Average Amount|Minimum Amount|Maximum Amount|
+-----------------+--------------+--------------+
|513.5020766335965|           0.0|      33501.33|
+-----------------+--------------+--------------+



In [0]:
billing_df.groupBy("status") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

+---------+-----+
|   status|count|
+---------+-----+
|completed| 9085|
|  pending| 1203|
|   failed|  832|
| refunded|  572|
+---------+-----+



In [0]:
analytics_df.select(
    avg("total_revenue").alias("Average Revenue"),
    min("total_revenue").alias("Minimum Revenue"),
    max("total_revenue").alias("Maximum Revenue")
).show()

+-----------------+---------------+---------------+
|  Average Revenue|Minimum Revenue|Maximum Revenue|
+-----------------+---------------+---------------+
|4815.098377192976|            0.0|       35632.86|
+-----------------+---------------+---------------+



In [0]:
crm_df.groupBy("city") \
      .count() \
      .orderBy("count", ascending=False) \
      .show()

+-------------+-----+
|         city|count|
+-------------+-----+
|       Mumbai|  800|
|    Bangalore|  743|
|        Delhi|  731|
|       Kanpur|  515|
|        Patna|  515|
|       Indore|  493|
|      Lucknow|  489|
|   Coimbatore|  486|
|    Ahmedabad|  486|
|      Kolkata|  479|
|         Pune|  470|
|       Jaipur|  467|
|     Vadodara|  464|
|       Bhopal|  463|
|    Hyderabad|  459|
|        Surat|  454|
|       Nagpur|  454|
|      Chennai|  449|
|Visakhapatnam|  447|
|        Thane|  438|
+-------------+-----+
only showing top 20 rows


In [0]:
avg_amount = billing_df.select(avg("amount")).first()[0]

if avg_amount > 500:
    print("🚨 Distribution Drift Detected")
else:
    print("✅ Distribution is Stable")

print("Average Billing Amount:", round(avg_amount, 2))

🚨 Distribution Drift Detected
Average Billing Amount: 513.5


In [0]:
crm_count = crm_df.count()
billing_count = billing_df.count()
analytics_count = analytics_df.count()

volume_difference = abs(crm_count - billing_count)
volume_drift = (volume_difference / crm_count) * 100

avg_amount = billing_df.selectExpr("avg(amount) as avg").first()["avg"]

schema_difference = list(
    set(crm_df.columns)
    .symmetric_difference(set(billing_df.columns))
)

In [0]:
from pyspark.sql import Row

drift_report = spark.createDataFrame([

    Row(
        crm_records=crm_count,
        billing_records=billing_count,
        analytics_records=analytics_count,

        volume_difference=volume_difference,
        volume_drift_percent=round(volume_drift,2),

        schema_drift=len(schema_difference),

        avg_billing_amount=round(avg_amount,2),

        distribution_status=
        "Drift Detected" if avg_amount > 500
        else "Stable"

    )

])

display(drift_report)

crm_records,billing_records,analytics_records,volume_difference,volume_drift_percent,schema_drift,avg_billing_amount,distribution_status
10195,11454,894,1259,12.35,8,524.17,Drift Detected


In [0]:
drift_report.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.drift_report")